In [ ]:
#-------------------------------------------------------------------------------
# Name:        04_Download_GEE_images
# Purpose:     Download the S2 clip under the grid
#
# Author:      Gijs van den Dool & Meggison Oritsemisan
#
# Created:     22/04/2024
# Copyright:   (c) Project Canopy Watch 2024
# Licence:     <Project Canopy Watch>
#-------------------------------------------------------------------------------
# Project Data Folder:
# https://drive.google.com/drive/folders/1dgFGkQ0Orjt8SbPtpOvPNb1A9y4Odeng?usp=drive_link

# Vector Masks:
# Output from the selection of ISL areas in a 1280x1280 bounding box, retaining
# only the bounding box

# Raster Masks:
# Input for the modelling, as a input for predictions

# Disclaimer: most code was developed by caleb, to work on the whole tile, the
# modification is that it performs the clip on the bounding area of the grid
# and exports in a 128x128 patch


In [ ]:
import os

import ee
import geemap

import datetime

import numpy as np
import pandas as pd
import geopandas as gpd

import rasterio
import geopandas as gpd

from datetime import datetime as DT
from tqdm.notebook import tqdm_notebook

In [ ]:
try:
        ee.Initialize()
except Exception as e:
        ee.Authenticate()
        ee.Initialize()

In [ ]:
# change these to match your configuration:
project_root = "/content/drive/MyDrive/Project Canopy Official Folder/Task 02 Data Preprocessing/Working Files/Experiment_ISL"
task_folder = "02_WorkingFiles"
file_name = "ImagePatch_ISL_02.geojson"  # Corrected file name

# Construct the file path using os.path.join()
file_path = os.path.join(project_root, task_folder,file_name)

# Test if the path is accessible
if os.path.isfile(file_path):
    print("File exists.")
else:
    print("File does not exist or is not accessible.")


File exists.


In [ ]:
# Define the file path to the shapefile
task_folder = "03_Output_Data"

# Create main output folder:
out_folder = os.path.join(project_root, task_folder, "02_download_Sentinel2_GEE/")

if os.path.exists(out_folder) == False:
  os.makedirs(out_folder)

  # Create output folder SR:
  out_folder1 = os.path.join(out_folder, "ImagePatch_SR/")
  os.makedirs(out_folder1)

  # Create output folder HM:
  out_folder2 = os.path.join(out_folder, "ImagePatch_HM/")
  os.makedirs(out_folder2)

else:
  print("folder already created")
  out_folder1 = os.path.join(out_folder, "ImagePatch_SR/")
  out_folder2 = os.path.join(out_folder, "ImagePatch_HM/")

folder already created


In [ ]:
gdf_ImagePatch = gpd.read_file(file_path)

print(len(gdf_ImagePatch)) # total lines in the AoI
print(gdf_ImagePatch.crs)  # current projection
gdf_ImagePatch.head(3)

18
EPSG:4326


,GID,geometry
0,GID_ISL_02_0_3,"POLYGON ((15.11366 2.28129, 15.12511 2.28124, ..."
1,GID_ISL_02_1_0,"POLYGON ((15.12525 2.31578, 15.13670 2.31573, ..."
2,GID_ISL_02_1_2,"POLYGON ((15.12516 2.29275, 15.13660 2.29271, ..."


In [ ]:
working_folder = "01_Input_Data"
file_name = "ISL_02.geojson" # might be different for other areas

# file_path = project_root + task_folder + woring_folder + file_name
file_path = os.path.join(project_root, working_folder, file_name)

# Test if the path is accessible
if os.path.isfile(file_path):
  gdf_ImageAoI = gpd.read_file(file_path)

  print(len(gdf_ImageAoI)) # total lines in the AoI
  print(gdf_ImageAoI.crs)  # current projection
  print(gdf_ImageAoI)
else:
  print("file missing")

1
EPSG:4326
  image_AoI    min_date    max_date  \
0    ISL_02  2022-03-30  2022-04-01   

                                                 url  image_date  \
0  https://apps.sentinel-hub.com/eo-browser/?zoom...  2022-03-31   

                                            geometry  
0  POLYGON ((15.11400 2.26899, 15.18289 2.26927, ...  


In [ ]:
def test_image_date(date_min, date_max):
  booImageFound = False

  s2_sr = ee.ImageCollection('COPERNICUS/S2_SR')\
    .filterDate(date_min, date_max)\
    .filterBounds(geometry) \
    .first() \

  try:
    #print(s2_sr.getInfo())
    sys_date = ee.Date(s2_sr.get('system:time_start'))

    time = sys_date.getInfo()
    #print(time.items())
    #print(time.get("value"))

    dt_time = datetime.datetime.fromtimestamp(time.get("value") // 1000)
    #print(dt_time)
    booImageFound = True

  except:
    #print("date is not valid, increasing window")
    booImageFound = False


  return booImageFound

In [ ]:
def getEVI(image):
    # Compute the EVI using an expression.
    EVI = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),
            'BLUE': image.select('B2').divide(10000)
        }).rename("B_EVI")

    image = image.addBands(EVI)

    return(image)

#   Calculate MSAVI2
def getMSAVI2(image):
    # Compute the MSAVI2 using an expression.
    MSAVI2 = image.expression(
        '(2 * NIR + 1 - sqrt(pow((2 * NIR + 1), 2) - 8 * (NIR - RED))) / 2', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),

        }).rename("B_MSAVI2")

    image = image.addBands(MSAVI2)

    return(image)

def getNDMI(image):
    # Compute the NDMI using an expression.
    NDMI = image.expression(
        '(B8 - B11) / (B8 + B11)', {
            'B8': image.select('B8').divide(10000),
            'B11': image.select('B11').divide(10000),

        }).rename("B_NDMI")

    #alternative:
    #ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')

    image = image.addBands(NDMI)

    return(image)

def getNBR(image):
    # Compute the NBR using an expression.
    NBR = image.expression(
        '(NIR - SWIR) / (NIR + SWIR)', {
            'NIR': image.select('B8').divide(10000),
            'SWIR': image.select('B12').divide(10000),

        }).rename("B_NBR")

    #alternative:
    #nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')

    image = image.addBands(NBR)

    return(image)

#   calculated as  NDBI= (SWIR-NIR/(SWIR+NIR)
def getNDBI(image):
    # Compute the NDBI using an expression.
    NDBI = image.expression(
        '(SWIR -NIR) / (SWIR + NIR)', {
            'NIR': image.select('B8').divide(10000),
            'SWIR': image.select('B11').divide(10000),

        }).rename("B_NDBI")

    #alternative:
    #ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')

    image = image.addBands(NDBI)

    return(image)

def getNDWI(image):
    # Compute the MSAVI2 using an expression.
    NDWI = image.expression(
        '(GREEN - NIR) / (GREEN + NIR)', {
            'NIR': image.select('B8').divide(10000),
            'GREEN': image.select('B3').divide(10000),

        }).rename("B_NDWI")

    image = image.addBands(NDWI)

    #alternative:
    #ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

    return(image)

#   calculated as NDVI = (NIR-RED) / (NIR+RED)

def getNDVI(image):
    # Compute the NDVI using an expression.
    NDVI = image.expression(
        '(NIR - RED) / (NIR + RED)', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),

        }).rename("B_NDVI")

    #alternative:
    #ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

    image = image.addBands(NDVI)

    return(image)

In [ ]:
def addDate(image):
    img_date = ee.Date(image.date())
    img_date = ee.Number.parse(img_date.format('YYYYMMdd'))
    return image.addBands(ee.Image(img_date).rename('date').toInt())

In [ ]:
print("steps: ", len(gdf_ImagePatch))

print(gdf_ImageAoI.min_date[0])

steps:  18
2022-03-30


In [ ]:
def adjust_geometry_for_fixed_size_output(centroid, target_pixels, scale):
    """
    Adjusts the geometry around a centroid to ensure the exported image matches the target pixel dimensions.

    :param centroid: Tuple of (longitude, latitude) for the AOI's centroid.
    :param target_pixels: The target size in pixels (e.g., 128 for a 128x128 image).
    :param scale: The scale in meters per pixel.
    :return: An ee.Geometry.Rectangle object for the adjusted region.
    """
# Adjust the target size in meters slightly to compensate for rounding up
    # adjustment_factor = 0.993  # Example factor; you may need to fine-tune this value
    target_size_meters = target_pixels * scale

    # Convert meters to degrees approximately (1 degree = approx. 111,319.9 meters)
    target_size_degrees = target_size_meters / 111319.9

    # Calculate the bounds of the rectangle
    half_size_degrees = target_size_degrees / 2
    min_lon = centroid[0] - half_size_degrees
    max_lon = centroid[0] + half_size_degrees
    min_lat = centroid[1] - half_size_degrees
    max_lat = centroid[1] + half_size_degrees

    # Create and return the geometry
    return ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])



In [ ]:
# Define output folders
folder_SR = out_folder1  # Sentinel Reflectance images
folder_HM = out_folder2  # Harmonized images

# Main loop over geodataframe entries
for k in tqdm_notebook(range(len(gdf_ImagePatch))):
    aoi = gdf_ImagePatch.iloc[k]
    strGID = aoi.GID

    # Construct file paths for potential output images
    file_path_SR = os.path.join(folder_SR, f"image_{strGID}.tif")
    file_path_HM = os.path.join(folder_HM, f"image_{strGID}.tif")

    # Check if the images already exist to skip processing
    if os.path.isfile(file_path_SR) and os.path.isfile(file_path_HM):
        print(f"Files for {strGID} already exist. Skipping processing.")
        continue

    # Proceed with calculations if files do not exist
    centroid = aoi.geometry.centroid.coords[0]
    fixed_pixels = 128
    scale = 10

    adjusted_geometry = adjust_geometry_for_fixed_size_output(
        centroid, target_pixels=fixed_pixels, scale=scale
    )

    minx, miny, maxx, maxy = aoi.geometry.bounds
    geometry = ee.Geometry.Rectangle([minx, miny, maxx, maxy])

    date_min = gdf_ImageAoI.min_date[0]
    date_max = gdf_ImageAoI.max_date[0]
    booTest = False  # Assume no valid image date is found initially

    for _ in range(10):  # Limit date testing to 10 iterations
        if test_image_date(date_min, date_max):
            booTest = True
            break

        # Adjust date_min and date_max by subtracting and adding one day respectively
        dt_object_min = datetime.datetime.strptime(date_min, '%Y-%m-%d') - datetime.timedelta(days=1)
        dt_object_max = datetime.datetime.strptime(date_max, '%Y-%m-%d') + datetime.timedelta(days=1)
        date_min = dt_object_min.strftime('%Y-%m-%d')
        date_max = dt_object_max.strftime('%Y-%m-%d')

    if booTest:
        # Processing image collections for both Sentinel Reflectance and Harmonized datasets
        collections = ['GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED', 'GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED']
        for collection, file_path in zip(collections, [file_path_SR, file_path_HM]):
            image_collection = ee.ImageCollection(collection) \
                .filterDate(date_min, date_max).filterBounds(adjusted_geometry) \
                .map(getNDVI).map(getEVI).map(getMSAVI2).map(getNDMI) \
                .map(getNBR).map(getNDBI).map(getNDWI).map(addDate).median()

            image_collection = image_collection.select('B.+')
            print(f"Processing {collection}: Number of bands after adding extra bands: {image_collection.bandNames().size().getInfo()}")

            try:
                geemap.ee_export_image(image_collection, filename=file_path, scale=10, region=adjusted_geometry.getInfo())
                print(image_collection.getInfo())
            except Exception as e:
                print(f"!!!!Issue!!!!.....check {strGID}........!!!!!!", e)

    else:
        print(f"No valid image found for {strGID}")

print("Processing complete")

  0%|          | 0/18 [00:00<?, ?it/s]

Number of bands after adding extra bands for SR: 19
Number of bands after adding extra bands for HM: 19
Generating URL ...
Please wait ...
Data downloaded to /Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_ISL_057/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_SR/image_GID_ISL_02_0_3.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_ISL_057/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_HM/image_GID_ISL_02_0_3.tif
Number of bands after adding extra bands for SR: 19
Number of bands after adding extra bands for HM: 19
Generating URL ...
Please wait ...
Data downloaded to /Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_ISL_057/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_SR/image_GID_ISL_02_1_0.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_ISL_057/03_Output_Data/02_download_S

In [ ]:
# Time between the first and last file processed on the GEE platform:

# image_GID_ISL_057_0_2
# ID: QPFWUBNDDA425XNM7H4WBRRY
# Phase: Completed
# Runtime: 13s (started 2023-11-12 12:34:25 +0100)
# Attempted 1 time
# Batch compute usage: 0.2500 EECU-seconds

# image_GID_ISL_057_57_10
# ID: QVWZN5WEODDANHMWZTWIWGKX
# Phase: Completed
# Runtime: 7s (started 2023-11-12 15:18:25 +0100)
# Attempted 1 time
# Batch compute usage: 0.1106 EECU-seconds

In [ ]:
# select bands:
from PIL import Image
import os

def resize_images_in_folder(source_folder, target_folder=None, size=(128, 128)):
    """
    Resize all TIFF images in the specified folder to 128x128 pixels.

    :param source_folder: The folder containing the original TIFF images.
    :param target_folder: The folder where resized images will be saved. If None, overwrite the source files.
    :param size: The target size for the images, default is (128, 128).
    """
    if target_folder is None:
        target_folder = source_folder
    else:
        # Create the target folder if it does not exist
        os.makedirs(target_folder, exist_ok=True)

    for filename in os.listdir(source_folder):
        if filename.endswith(".tif") or filename.endswith(".tiff"):
            # Construct the full file path
            file_path = os.path.join(source_folder, filename)
            # Open the image
            with Image.open(file_path) as img:
                # Resize the image/
                resized_img = img.resize(size, Image.ANTIALIAS)
                # Construct the save path
                save_path = os.path.join(target_folder, filename)
                # Save the resized image
                resized_img.save(save_path)
                print(f"Resized and saved: {save_path}")

# Example usage
source_folder = '/Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_ISL_057/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_HM'
target_folder = '/Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_ISL_057/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_HM'  # Can be the same as source_folder or a new one
resize_images_in_folder(source_folder, target_folder)


More samples per pixel than can be decoded: 19
More samples per pixel than can be decoded: 19


UnidentifiedImageError: cannot identify image file '/Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_ISL_057/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_HM/image_GID_ISL_46_3_1.tif'

In [ ]:
geometry

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Polygon",
    "arguments": {
      "coordinates": {
        "constantValue": [
          [
            [
              17.64155112118302,
              3.236147820820384
            ],
            [
              17.64155112118302,
              3.224602031778878
            ],
            [
              17.653038114693828,
              3.224602031778878
            ],
            [
              17.653038114693828,
              3.236147820820384
            ]
          ]
        ]
      },
      "evenOdd": {
        "constantValue": true
      }
    }
  }
})

In [ ]:
# resetting the paths, to be sure that the files are going to the right location
# on the shared drive

# storing the paths for export:
out_folder1 = "03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_SR/"
out_folder1 = project_root + task_folder + out_folder1
print(out_folder1)

out_folder2 = "03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_HM/"
out_folder2 = project_root + task_folder + out_folder2
print(out_folder2)

./drive/MyDrive/OmdenaProjectCanopy/__SharedProjectCanopy/task-04-data-preprocessing/Experiment_ISL_057/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_SR/
./drive/MyDrive/OmdenaProjectCanopy/__SharedProjectCanopy/task-04-data-preprocessing/Experiment_ISL_057/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_HM/


In [ ]:
os.path.exists(out_folder2)

False

In [ ]:
# moving the files from the local drive to the shared drive
# This can only happen when all files are processed

# ToDo: include a test

import os
import shutil

folder_SR = "./drive/MyDrive/tmp_SR"
folder_HM = "./drive/MyDrive/tmp_HM"
os.path.exists(folder_SR)

False

In [ ]:
images = [f for f in os.listdir(folder_HM)]
print(len(images))
if len(images) > 0:
  print(images[0])
else:
  print("no files")

FileNotFoundError: [Errno 2] No such file or directory: './drive/MyDrive/tmp_HM'

In [ ]:
for image in images:
  old_path = folder_HM + "/" + image
  new_path = out_folder2 + image
  shutil.move(old_path, new_path)

  old_path = folder_SR + "/" +  image
  new_path = out_folder1 + image
  shutil.move(old_path, new_path)



In [ ]:
images = [f for f in os.listdir(out_folder1)]
print(len(images))
if len(images) > 0:
  print(images[0])
else:
  print("no file")